# repovideo — Colab GPU Worker

This notebook installs your **actual repovideo project** from GitHub and runs the GPU-heavy parts on Colab's A100/T4. No copy-pasted code — when you update the repo, the notebook gets the updates too.

### Quick start
1. Run **Setup** (cells 1-3: install, HF login, GPU check)
2. Pick a path:
   - **Section A** — Train a LoRA (**A1** defaults to **pusa** + **480P** + **50 steps** + **9 frames** for a *fast* smoke test; **YouTube-Commons** is much slower)
   - **Section B** — Generate an anecdote video
   - **Section C** — Download results to your Mac
   - **Section D** — Run as automated worker for `--colab` flag

In [ ]:
#@title 1. Install repovideo from your GitHub repo
#@markdown This clones your repo and installs it as a real Python package.
#@markdown Change the URL if you've forked it.
#@markdown
#@markdown **If the repo is private**, paste a GitHub personal access token below.
#@markdown Create one at https://github.com/settings/tokens (repo scope).

REPO_URL = 'https://github.com/ibrahim-ansari-code/repo---video.git' #@param {type:"string"}
GITHUB_TOKEN = '' #@param {type:"string"}

import os, subprocess, sys

clone_dir = '/tmp/repovideo'

# Build authenticated URL if token provided
clone_url = REPO_URL
if GITHUB_TOKEN:
    clone_url = REPO_URL.replace('https://github.com/', f'https://{GITHUB_TOKEN}@github.com/')

# Clone or update
if os.path.isdir(clone_dir):
    print('Repo already cloned, pulling latest...')
    r = subprocess.run(['git', '-C', clone_dir, 'pull'], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print(f'Cloning {REPO_URL} ...')
    r = subprocess.run(['git', 'clone', '--depth', '1', clone_url, clone_dir],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'❌ Clone failed:\n{r.stderr}')
        if 'not found' in r.stderr.lower() or '128' in str(r.returncode):
            print('\nIs the repo private? Paste a GitHub token in GITHUB_TOKEN above.')
        sys.exit(1)
    print('Cloned ✅')

# Install as editable package with train extras (includes yt-dlp for YouTube-Commons)
!pip install -q "{clone_dir}[train]"
!pip install -q decord yt-dlp

print('\n✅ repovideo installed')
!repovideo --help

In [ ]:
#@title 2. Log in to Hugging Face (required for Wan2.1 — gated model)
#@markdown Go to https://huggingface.co/Wan-AI/Wan2.1-I2V-14B-480P-Diffusers and click
#@markdown **"Agree and access repository"** first.
#@markdown
#@markdown Then paste your HF token (https://huggingface.co/settings/tokens):

HF_TOKEN = '' #@param {type:"string"}

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('✅ Logged in to Hugging Face')
else:
    try:
        from huggingface_hub import HfApi
        info = HfApi().whoami()
        print(f'Already logged in as: {info["name"]}')
    except Exception:
        print('⚠️  Not logged in. Paste your HF token above.')
        print('   Wan2.1 is a gated model — you need to accept the license and log in.')
        print('   1. Visit https://huggingface.co/Wan-AI/Wan2.1-I2V-14B-480P-Diffusers')
        print('   2. Click "Agree and access repository"')
        print('   3. Create a token at https://huggingface.co/settings/tokens')
        print('   4. Paste it in HF_TOKEN above and re-run this cell')

In [ ]:
#@title 3. Check GPU + see available datasets
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1024**3
    print(f'GPU:  {gpu}')
    print(f'VRAM: {vram:.1f} GB')
    RECOMMENDED = '14B' if vram >= 38 else '480P'
    print(f'→ Recommended model: {RECOMMENDED}')
else:
    print('⚠️  No GPU. Go to Runtime → Change runtime type → T4 GPU')
    RECOMMENDED = '480P'

print()
!repovideo list-datasets

---
## Section A: Train a LoRA

**Speed:** First run is slow because the **14B Wan** model is huge (download + staged VAE/text/CLIP/transformer). **YouTube-Commons** adds **yt-dlp** per clip (often 10–40+ min for 8 videos). **Fastest smoke test:** keep defaults below (**pusa**, **480P**, **50** steps, **9** frames, **8** samples).

Two options:
- **A1**: Use a built-in HF dataset (no uploads needed)
- **A2**: Upload your own reference videos/images

**Dataset tiers (best → worst for I2V LoRA):**
| Dataset | Tier | What it has | What it trains |
|---------|------|-------------|----------------|
| `pusa` | **VIDEO** | 3,860 Wan-generated video + caption + first frame | Motion + style with I2V conditioning (**recommended**) |
| `cinematic` | VIDEO | Film clips + scene descriptions | Motion + style |
| `pexels` | VIDEO | Stock footage via URLs | Motion + style |
| `youtube-commons` | VIDEO | CC-BY YouTube (parquet + **yt-dlp** clips) | Motion + style; use **8–16** samples first |
| `tip-i2v` | I2V | Image + text prompt pairs (no video output) | Style only (denoising autoencoder on stills) |
| `developer` | IMAGE | Tech/dev scene thumbnails | Style only (no motion) |
| `dramatic` | IMAGE | Moody/dramatic thumbnails | Style only (no motion) |

In [ ]:
#@title A1: Train on a built-in dataset (recommended to start)
#@markdown Defaults = **fast smoke test**. **youtube-commons** = slow (yt-dlp); use **pusa** first.
#@markdown After a run works: raise **TRAIN_STEPS** (200–500), **NUM_FRAMES** (17), try **14B** if you have VRAM.

DATASET = 'pusa' #@param ["pusa", "youtube-commons", "cinematic", "pexels", "tip-i2v", "developer", "dramatic"]
MODEL_SIZE = '480P' #@param ["480P", "14B"]
TRAIN_STEPS = 50 #@param {type:"integer"}
NUM_FRAMES = 9 #@param {type:"integer"}
NUM_SAMPLES = 8 #@param {type:"integer"}
#@markdown **NUM_SAMPLES**: **8** for speed; **50–200** when training for real.
#@markdown **NUM_FRAMES**: **9** = faster encoding; **17** = closer to full I2V quality.

import os
import shutil
import subprocess
import sys

from src.config import ensure_dirs
from src.anecdote.datasets import download_dataset, get_dataset_tier
from src.anecdote.lora_trainer import LoRATrainingConfig, train_lora

ensure_dirs()

if DATASET == 'youtube-commons':
    if not shutil.which('yt-dlp'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)
    if not shutil.which('yt-dlp'):
        raise RuntimeError('yt-dlp not found. Re-run cell 1, then this cell.')
    os.environ.setdefault('REPOVIDEO_YTC_CLIP_SECONDS', '8')
    print('=' * 60)
    print('YouTube-Commons — expect LONG downloads (yt-dlp + YouTube).')
    print(f'  NUM_SAMPLES={NUM_SAMPLES}  TRAIN_STEPS={TRAIN_STEPS}  clip≈8s')
    print('  Many URLs fail; that is normal. For speed: stop & switch DATASET to pusa.')
    print('=' * 60)

tier = get_dataset_tier(DATASET)
print(f'Dataset: {DATASET} (tier: {tier})')
print(f'Downloading / preparing data ({NUM_SAMPLES} samples max)...')
data_dir = download_dataset(DATASET, max_samples=NUM_SAMPLES)

print(f'\nTraining LoRA ({TRAIN_STEPS} steps, Wan2.1 {MODEL_SIZE}, {NUM_FRAMES} frames)...')
if tier == 'image':
    print('⚠️  Image-only dataset: will learn style but NOT motion.')
    print('   Use "pusa" or "cinematic" for full video diffusion training.\n')

config = LoRATrainingConfig(
    reference_dir=data_dir,
    dataset_name=DATASET,
    dataset_max_samples=NUM_SAMPLES,
    lora_name=DATASET,
    model_size=MODEL_SIZE,
    num_train_steps=TRAIN_STEPS,
    num_frames=NUM_FRAMES,
)
LORA_DIR = train_lora(config)
print(f'\n✅ LoRA trained: {LORA_DIR}')

In [ ]:
#@title A2: Train on your own images (alternative to A1)
#@markdown Upload your reference images, then train.

import os
from pathlib import Path
from google.colab import files as colab_files
from src.config import ensure_dirs
from src.anecdote.lora_trainer import LoRATrainingConfig, train_lora

ensure_dirs()

LORA_NAME = 'my_style' #@param {type:"string"}
MODEL_SIZE = '14B' #@param ["14B", "480P"]
TRAIN_STEPS = 500 #@param {type:"integer"}

upload_dir = Path('/content/reference_data')
upload_dir.mkdir(exist_ok=True)

print('Upload your reference images (.png, .jpg) or videos (.mp4):')
uploaded = colab_files.upload()
for name, data in uploaded.items():
    (upload_dir / name).write_bytes(data)
    print(f'  Saved {name}')

print(f'\nTraining LoRA on {len(uploaded)} files...')
config = LoRATrainingConfig(
    reference_dir=upload_dir,
    lora_name=LORA_NAME,
    model_size=MODEL_SIZE,
    num_train_steps=TRAIN_STEPS,
)
LORA_DIR = train_lora(config)
print(f'\n✅ LoRA trained: {LORA_DIR}')

---
## Section B: Generate the anecdote video

Creates the AI intro for your demo:
1. SDXL generates keyframe images from your prompts
2. Wan2.1 animates them into video clips (using the LoRA if you trained one above)
3. FFmpeg concatenates into `anecdote.mp4`

In [ ]:
#@title B1: Configure prompts
#@markdown Edit these to match the project you're making a demo for.

keyframe_prompts = [
    'A developer sitting at a cluttered desk late at night, multiple browser tabs open on dual monitors, coffee cup empty, warm desk lamp lighting, photorealistic, cinematic shallow depth of field, 4K quality',
    'Close-up of a laptop screen showing a wall of error messages and red warning icons, reflection of a tired developer in the screen, dramatic lighting from the monitor, ultra detailed',
    'A developer throwing their hands up in frustration, papers and sticky notes scattered on a desk, dramatic side lighting, editorial photography style, 8K resolution',
]

motion_prompts = [
    'Camera slowly zooms in on the developer face, subtle head shake',
    'Screen content slowly scrolling down, flickering light from monitor',
    'Hands move upward in frustration, slight camera shake',
]

USE_MODEL_SIZE = '14B' #@param ["14B", "480P"]
USE_LORA = True #@param {type:"boolean"}

lora_path = LORA_DIR if (USE_LORA and 'LORA_DIR' in dir()) else None
print(f'Model: Wan2.1 {USE_MODEL_SIZE}')
print(f'LoRA:  {lora_path or "none (base model)"}')
print(f'Keyframes: {len(keyframe_prompts)}')

In [ ]:
#@title B2: Generate keyframe images (SDXL)
from src.anecdote.image_gen import generate_keyframes
from pathlib import Path

print('Generating keyframe images with SDXL...')
image_paths = generate_keyframes(keyframe_prompts, Path('/content/keyframes'))

print(f'\n✅ Generated {len(image_paths)} keyframes')

from IPython.display import display
from PIL import Image
for p in image_paths:
    display(Image.open(p).resize((640, 360)))

In [ ]:
#@title B3: Animate into video clips (Wan2.1 I2V + your LoRA)
from src.anecdote.video_gen import generate_video_from_images
from pathlib import Path

anecdote_path = Path('/content/anecdote.mp4')

print(f'Generating video clips with Wan2.1 {USE_MODEL_SIZE}...')
if lora_path:
    print(f'Using LoRA: {lora_path}')

generate_video_from_images(
    image_paths=image_paths,
    motion_prompts=motion_prompts,
    output_path=anecdote_path,
    lora_path=lora_path,
    model_size=USE_MODEL_SIZE,
)

if anecdote_path.exists():
    size_mb = anecdote_path.stat().st_size / 1024 / 1024
    print(f'\n✅ anecdote.mp4 ready ({size_mb:.1f} MB)')
else:
    print('⚠️  Generation failed — check the logs above')

---
## Section C: Download results to your Mac

In [ ]:
#@title C1: Direct download (through your browser)
from google.colab import files as colab_files
from pathlib import Path
import shutil

# Download the anecdote video
if Path('/content/anecdote.mp4').exists():
    print('Downloading anecdote.mp4...')
    colab_files.download('/content/anecdote.mp4')

# Download LoRA weights as a zip
if 'LORA_DIR' in dir() and Path(str(LORA_DIR)).exists():
    zip_name = f'/content/{Path(str(LORA_DIR)).name}_lora'
    shutil.make_archive(zip_name, 'zip', str(LORA_DIR))
    print(f'Downloading LoRA weights...')
    colab_files.download(f'{zip_name}.zip')

In [ ]:
#@title C2: Save to Google Drive
import shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

dest = Path('/content/drive/MyDrive/repovideo_results')
dest.mkdir(parents=True, exist_ok=True)

if Path('/content/anecdote.mp4').exists():
    shutil.copy2('/content/anecdote.mp4', dest / 'anecdote.mp4')
    print(f'✅ anecdote.mp4 → Google Drive')

if 'LORA_DIR' in dir() and Path(str(LORA_DIR)).exists():
    lora_dest = dest / 'loras' / Path(str(LORA_DIR)).name
    shutil.copytree(str(LORA_DIR), str(lora_dest), dirs_exist_ok=True)
    print(f'✅ LoRA weights → Google Drive')

print(f'\nAll saved to: {dest}')
print('Files will sync to your Mac via Google Drive for Desktop.')

---
## Using the results on your Mac

```bash
# Option 1: Just plug in the anecdote video you downloaded
repovideo generate https://github.com/user/repo \
    --anecdote-file ~/Downloads/anecdote.mp4

# Option 2: Install the LoRA locally for future runs
unzip ~/Downloads/developer_lora.zip -d ~/.repovideo/loras/developer/
repovideo generate https://github.com/user/repo \
    --lora-name developer --model-size 480P

# Option 3: Full automation via Google Drive
# (run Section D in Colab, then on your Mac:)
repovideo generate https://github.com/user/repo --colab
```

---
## Section D: Automated job watcher (for `--colab` flag)

Leave this running in Colab. On your Mac, use `repovideo generate <url> --colab`
and the GPU work gets dispatched here automatically via Google Drive.

In [ ]:
#@title D1: Start job watcher
import json
import shutil
import subprocess
import time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from src.config import ensure_dirs
from src.anecdote.image_gen import generate_keyframes
from src.anecdote.video_gen import generate_video_from_images
from src.anecdote.lora_trainer import LoRATrainingConfig, train_lora

ensure_dirs()
JOBS_DIR = Path('/content/drive/MyDrive/repovideo_jobs')
JOBS_DIR.mkdir(parents=True, exist_ok=True)

def process_job(job_dir):
    job_dir = Path(job_dir)
    config = json.loads((job_dir / 'config.json').read_text())
    results_dir = job_dir / 'results'
    results_dir.mkdir(exist_ok=True)
    job_type = config['job_type']
    print(f'  Processing: {job_type}')

    if job_type == 'generate_anecdote':
        kf_dir = results_dir / 'keyframes'
        img_paths = generate_keyframes(config['keyframe_prompts'], kf_dir)
        anecdote_path = results_dir / 'anecdote.mp4'
        generate_video_from_images(
            img_paths, config['motion_prompts'], anecdote_path,
            lora_path=Path(config['lora_path']) if config.get('lora_path') else None,
            model_size=config.get('model_size', '14B'),
        )

    elif job_type == 'train_lora':
        tc = LoRATrainingConfig(
            reference_dir=job_dir / 'reference_data',
            lora_name=config.get('lora_name', 'custom_style'),
            model_size=config.get('model_size', '14B'),
            num_train_steps=config.get('num_train_steps', 500),
        )
        lora_dir = train_lora(tc)
        shutil.copytree(str(lora_dir), str(results_dir / 'lora'), dirs_exist_ok=True)

    (job_dir / 'status.txt').write_text('completed')
    print(f'  ✅ Done')

print(f'Watching {JOBS_DIR} for jobs...')
print('On your Mac: repovideo generate <url> --colab\n')

while True:
    for jd in sorted(JOBS_DIR.iterdir()):
        if not jd.is_dir(): continue
        sf = jd / 'status.txt'
        if not sf.exists() or sf.read_text().strip() != 'pending': continue
        try:
            process_job(jd)
        except Exception as e:
            print(f'  ❌ Failed: {e}')
            (jd / 'status.txt').write_text(f'failed: {e}')
    time.sleep(15)